# RQ2 — U-Net shot-count study on DIVA-HisDB (autonomous Colab run)

ConvNeXt-Tiny U-Net trained on k ∈ {1, 3, 5, 10, 15} labeled pages chosen by six page-selection methods (random, grayscale variance, PCA/ICA max–min and centroid) for CB55, CS18 and CS863. The 90 (subset, method, k) cells share **77 distinct page sets** (the RQ2 shot-selection manifest, embedded below, identical to the two-stage and Mask R-CNN studies). Each page set is trained with two initializations — ImageNet (`convnext_tiny.in12k_ft_in1k`) and DINOv3 LVD-1689M (`convnext_tiny.dinov3_lvd1689m`), the encoders of the k = 20 thesis rows — giving **154 runs**.

Recipe: as the k = 20 rows (100 epochs, early-stopping patience 30 epochs, lr 1e-3 / backbone 1e-4, batch 4, crop 448, weight decay 1e-4, AMP, seed 42), except that — as in the two-stage and Mask R-CNN shot studies — validation and checkpoint selection run **every 5th epoch** (and at the last one), with bf16 autocast and batched sliding windows. Evaluation is unchanged: seam-carving postprocessing and the Java DIVA evaluator on the public test split.

Everything is fetched by the notebook itself (code, DIVA-HisDB from Zenodo, timm weights, evaluator jar). **Runtime → Change runtime type → A100 GPU**, then *Run all*. Parallel runs are derived from GPU memory and host RAM. Results go to Drive (`MyDrive/thesis_rq2_unet_diva/`); the notebook is resumable (finished runs are skipped, checkpoints on Drive are re-used, runs still alive from an interrupted execution are waited for). Interrupting the cell cancels the queue but lets started runs finish; before re-running, use Runtime → Restart session.

## 1. Configuration

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json

DRIVE_OUT = Path("/content/drive/MyDrive/thesis_rq2_unet_diva")   # persistent results
REPO      = Path("/content/repo")                                   # code + data (local VM)
DATA      = REPO / "00_data" / "DIVA-HisDB"
JAR       = Path("/content/LineSegmentationEvaluator.jar")
ZENODO    = "https://zenodo.org/api/records/19127869/files/DIVAHisDB.tar.gz/content"
JAR_URL   = "https://github.com/DIVA-DIA/DIVA_Line_Segmentation_Evaluator/raw/master/out/artifacts/LineSegmentationEvaluator.jar"

SUBSETS = ["CB55", "CS18", "CS863"]
# (short tag, encoder name as used by the project, initialization) -- encoders of the k=20 rows
CONFIGS = [
    ("convnext_tiny", "tu-convnext_tiny.in12k_ft_in1k",   "imagenet"),
    ("convnext_tiny", "tu-convnext_tiny.dinov3_lvd1689m", "dinov3"),
]
EPOCHS, PATIENCE, LR, LR_BACKBONE = 100, 30, 1e-3, 1e-4
BATCH_SIZE, WEIGHT_DECAY, SEED = 4, 1e-4, 42
VAL_EVERY, VAL_WINDOW_BATCH = 5, 16   # validation every 5th epoch (bf16, batched windows)
GB_PER_RUN  = 10    # peak GPU memory of one run incl. batched validation
RAM_PER_RUN = 9     # host RAM of one run incl. cached pages
PARALLEL    = None  # None = as many runs as GPU memory and RAM allow

# RQ2 shot-selection manifest: 77 distinct page sets ("jobs") and the 90 cells using them.
MANIFEST = json.loads(r"""{
 "jobs": [
  {
   "job": "CB55_k1_8c26820d",
   "subset": "CB55",
   "k": 1,
   "pages": [
    "e-codices_fmb-cb-0055_0043v_max"
   ]
  },
  {
   "job": "CB55_k1_7a1608db",
   "subset": "CB55",
   "k": 1,
   "pages": [
    "e-codices_fmb-cb-0055_0005v_max"
   ]
  },
  {
   "job": "CB55_k1_ddbc80ee",
   "subset": "CB55",
   "k": 1,
   "pages": [
    "e-codices_fmb-cb-0055_0008v_max"
   ]
  },
  {
   "job": "CB55_k1_2c752f4a",
   "subset": "CB55",
   "k": 1,
   "pages": [
    "e-codices_fmb-cb-0055_0011v_max"
   ]
  },
  {
   "job": "CB55_k3_e5547c3a",
   "subset": "CB55",
   "k": 3,
   "pages": [
    "e-codices_fmb-cb-0055_0008v_max",
    "e-codices_fmb-cb-0055_0031v_max",
    "e-codices_fmb-cb-0055_0043v_max"
   ]
  },
  {
   "job": "CB55_k3_f2700177",
   "subset": "CB55",
   "k": 3,
   "pages": [
    "e-codices_fmb-cb-0055_0008v_max",
    "e-codices_fmb-cb-0055_0011v_max",
    "e-codices_fmb-cb-0055_0087r_max"
   ]
  },
  {
   "job": "CB55_k3_ce938512",
   "subset": "CB55",
   "k": 3,
   "pages": [
    "e-codices_fmb-cb-0055_0008v_max",
    "e-codices_fmb-cb-0055_0011v_max",
    "e-codices_fmb-cb-0055_0067v_max"
   ]
  },
  {
   "job": "CB55_k3_2017fce1",
   "subset": "CB55",
   "k": 3,
   "pages": [
    "e-codices_fmb-cb-0055_0008v_max",
    "e-codices_fmb-cb-0055_0060r_max",
    "e-codices_fmb-cb-0055_0067v_max"
   ]
  },
  {
   "job": "CB55_k3_458ebb5d",
   "subset": "CB55",
   "k": 3,
   "pages": [
    "e-codices_fmb-cb-0055_0008v_max",
    "e-codices_fmb-cb-0055_0011r_max",
    "e-codices_fmb-cb-0055_0011v_max"
   ]
  },
  {
   "job": "CB55_k3_9c4b2a61",
   "subset": "CB55",
   "k": 3,
   "pages": [
    "e-codices_fmb-cb-0055_0005v_max",
    "e-codices_fmb-cb-0055_0011v_max",
    "e-codices_fmb-cb-0055_0043v_max"
   ]
  },
  {
   "job": "CB55_k5_4d2a742b",
   "subset": "CB55",
   "k": 5,
   "pages": [
    "e-codices_fmb-cb-0055_0008v_max",
    "e-codices_fmb-cb-0055_0031v_max",
    "e-codices_fmb-cb-0055_0043v_max",
    "e-codices_fmb-cb-0055_0058v_max",
    "e-codices_fmb-cb-0055_0073v_max"
   ]
  },
  {
   "job": "CB55_k5_86a6d5af",
   "subset": "CB55",
   "k": 5,
   "pages": [
    "e-codices_fmb-cb-0055_0005v_max",
    "e-codices_fmb-cb-0055_0011v_max",
    "e-codices_fmb-cb-0055_0026v_max",
    "e-codices_fmb-cb-0055_0060r_max",
    "e-codices_fmb-cb-0055_0067v_max"
   ]
  },
  {
   "job": "CB55_k5_82d0bdca",
   "subset": "CB55",
   "k": 5,
   "pages": [
    "e-codices_fmb-cb-0055_0005v_max",
    "e-codices_fmb-cb-0055_0008v_max",
    "e-codices_fmb-cb-0055_0011r_max",
    "e-codices_fmb-cb-0055_0011v_max",
    "e-codices_fmb-cb-0055_0067v_max"
   ]
  },
  {
   "job": "CB55_k5_9072af77",
   "subset": "CB55",
   "k": 5,
   "pages": [
    "e-codices_fmb-cb-0055_0005v_max",
    "e-codices_fmb-cb-0055_0011v_max",
    "e-codices_fmb-cb-0055_0043v_max",
    "e-codices_fmb-cb-0055_0060r_max",
    "e-codices_fmb-cb-0055_0067v_max"
   ]
  },
  {
   "job": "CB55_k5_45043204",
   "subset": "CB55",
   "k": 5,
   "pages": [
    "e-codices_fmb-cb-0055_0005v_max",
    "e-codices_fmb-cb-0055_0008v_max",
    "e-codices_fmb-cb-0055_0011r_max",
    "e-codices_fmb-cb-0055_0011v_max",
    "e-codices_fmb-cb-0055_0060r_max"
   ]
  },
  {
   "job": "CB55_k5_87c37879",
   "subset": "CB55",
   "k": 5,
   "pages": [
    "e-codices_fmb-cb-0055_0005v_max",
    "e-codices_fmb-cb-0055_0011v_max",
    "e-codices_fmb-cb-0055_0031v_max",
    "e-codices_fmb-cb-0055_0043v_max",
    "e-codices_fmb-cb-0055_0073v_max"
   ]
  },
  {
   "job": "CB55_k10_69b45a69",
   "subset": "CB55",
   "k": 10,
   "pages": [
    "e-codices_fmb-cb-0055_0008v_max",
    "e-codices_fmb-cb-0055_0026v_max",
    "e-codices_fmb-cb-0055_0027v_max",
    "e-codices_fmb-cb-0055_0031v_max",
    "e-codices_fmb-cb-0055_0043v_max",
    "e-codices_fmb-cb-0055_0058v_max",
    "e-codices_fmb-cb-0055_0059r_max",
    "e-codices_fmb-cb-0055_0060r_max",
    "e-codices_fmb-cb-0055_0073v_max",
    "e-codices_fmb-cb-0055_0074v_max"
   ]
  },
  {
   "job": "CB55_k10_d688d043",
   "subset": "CB55",
   "k": 10,
   "pages": [
    "e-codices_fmb-cb-0055_0005v_max",
    "e-codices_fmb-cb-0055_0011v_max",
    "e-codices_fmb-cb-0055_0025r_max",
    "e-codices_fmb-cb-0055_0026v_max",
    "e-codices_fmb-cb-0055_0031v_max",
    "e-codices_fmb-cb-0055_0058v_max",
    "e-codices_fmb-cb-0055_0059v_max",
    "e-codices_fmb-cb-0055_0060r_max",
    "e-codices_fmb-cb-0055_0072v_max",
    "e-codices_fmb-cb-0055_0074v_max"
   ]
  },
  {
   "job": "CB55_k10_5a5195ef",
   "subset": "CB55",
   "k": 10,
   "pages": [
    "e-codices_fmb-cb-0055_0005v_max",
    "e-codices_fmb-cb-0055_0008v_max",
    "e-codices_fmb-cb-0055_0011r_max",
    "e-codices_fmb-cb-0055_0011v_max",
    "e-codices_fmb-cb-0055_0025r_max",
    "e-codices_fmb-cb-0055_0058v_max",
    "e-codices_fmb-cb-0055_0060r_max",
    "e-codices_fmb-cb-0055_0067v_max",
    "e-codices_fmb-cb-0055_0084r_max",
    "e-codices_fmb-cb-0055_0087r_max"
   ]
  },
  {
   "job": "CB55_k10_ad7bd714",
   "subset": "CB55",
   "k": 10,
   "pages": [
    "e-codices_fmb-cb-0055_0005v_max",
    "e-codices_fmb-cb-0055_0011v_max",
    "e-codices_fmb-cb-0055_0025r_max",
    "e-codices_fmb-cb-0055_0026v_max",
    "e-codices_fmb-cb-0055_0027v_max",
    "e-codices_fmb-cb-0055_0031v_max",
    "e-codices_fmb-cb-0055_0058v_max",
    "e-codices_fmb-cb-0055_0059v_max",
    "e-codices_fmb-cb-0055_0060r_max",
    "e-codices_fmb-cb-0055_0074v_max"
   ]
  },
  {
   "job": "CB55_k10_69341241",
   "subset": "CB55",
   "k": 10,
   "pages": [
    "e-codices_fmb-cb-0055_0005v_max",
    "e-codices_fmb-cb-0055_0008v_max",
    "e-codices_fmb-cb-0055_0011r_max",
    "e-codices_fmb-cb-0055_0011v_max",
    "e-codices_fmb-cb-0055_0031v_max",
    "e-codices_fmb-cb-0055_0043v_max",
    "e-codices_fmb-cb-0055_0058v_max",
    "e-codices_fmb-cb-0055_0059r_max",
    "e-codices_fmb-cb-0055_0073v_max",
    "e-codices_fmb-cb-0055_0074v_max"
   ]
  },
  {
   "job": "CB55_k15_668f4fda",
   "subset": "CB55",
   "k": 15,
   "pages": [
    "e-codices_fmb-cb-0055_0005v_max",
    "e-codices_fmb-cb-0055_0008v_max",
    "e-codices_fmb-cb-0055_0026v_max",
    "e-codices_fmb-cb-0055_0027v_max",
    "e-codices_fmb-cb-0055_0031v_max",
    "e-codices_fmb-cb-0055_0043v_max",
    "e-codices_fmb-cb-0055_0058v_max",
    "e-codices_fmb-cb-0055_0059r_max",
    "e-codices_fmb-cb-0055_0059v_max",
    "e-codices_fmb-cb-0055_0060r_max",
    "e-codices_fmb-cb-0055_0067v_max",
    "e-codices_fmb-cb-0055_0072v_max",
    "e-codices_fmb-cb-0055_0073v_max",
    "e-codices_fmb-cb-0055_0074v_max",
    "e-codices_fmb-cb-0055_0084r_max"
   ]
  },
  {
   "job": "CB55_k15_5c363272",
   "subset": "CB55",
   "k": 15,
   "pages": [
    "e-codices_fmb-cb-0055_0005v_max",
    "e-codices_fmb-cb-0055_0008v_max",
    "e-codices_fmb-cb-0055_0011v_max",
    "e-codices_fmb-cb-0055_0025r_max",
    "e-codices_fmb-cb-0055_0026v_max",
    "e-codices_fmb-cb-0055_0027v_max",
    "e-codices_fmb-cb-0055_0031v_max",
    "e-codices_fmb-cb-0055_0058v_max",
    "e-codices_fmb-cb-0055_0059v_max",
    "e-codices_fmb-cb-0055_0060r_max",
    "e-codices_fmb-cb-0055_0067v_max",
    "e-codices_fmb-cb-0055_0072v_max",
    "e-codices_fmb-cb-0055_0073v_max",
    "e-codices_fmb-cb-0055_0074v_max",
    "e-codices_fmb-cb-0055_0087r_max"
   ]
  },
  {
   "job": "CB55_k15_634568b4",
   "subset": "CB55",
   "k": 15,
   "pages": [
    "e-codices_fmb-cb-0055_0005v_max",
    "e-codices_fmb-cb-0055_0008v_max",
    "e-codices_fmb-cb-0055_0011r_max",
    "e-codices_fmb-cb-0055_0011v_max",
    "e-codices_fmb-cb-0055_0025r_max",
    "e-codices_fmb-cb-0055_0031v_max",
    "e-codices_fmb-cb-0055_0058v_max",
    "e-codices_fmb-cb-0055_0059r_max",
    "e-codices_fmb-cb-0055_0059v_max",
    "e-codices_fmb-cb-0055_0060r_max",
    "e-codices_fmb-cb-0055_0067v_max",
    "e-codices_fmb-cb-0055_0072v_max",
    "e-codices_fmb-cb-0055_0074v_max",
    "e-codices_fmb-cb-0055_0084r_max",
    "e-codices_fmb-cb-0055_0087r_max"
   ]
  },
  {
   "job": "CB55_k15_7e3c513c",
   "subset": "CB55",
   "k": 15,
   "pages": [
    "e-codices_fmb-cb-0055_0005v_max",
    "e-codices_fmb-cb-0055_0008v_max",
    "e-codices_fmb-cb-0055_0011r_max",
    "e-codices_fmb-cb-0055_0011v_max",
    "e-codices_fmb-cb-0055_0027v_max",
    "e-codices_fmb-cb-0055_0031v_max",
    "e-codices_fmb-cb-0055_0043v_max",
    "e-codices_fmb-cb-0055_0058v_max",
    "e-codices_fmb-cb-0055_0059r_max",
    "e-codices_fmb-cb-0055_0059v_max",
    "e-codices_fmb-cb-0055_0060r_max",
    "e-codices_fmb-cb-0055_0072v_max",
    "e-codices_fmb-cb-0055_0073v_max",
    "e-codices_fmb-cb-0055_0074v_max",
    "e-codices_fmb-cb-0055_0084r_max"
   ]
  },
  {
   "job": "CS18_k1_789b4135",
   "subset": "CS18",
   "k": 1,
   "pages": [
    "e-codices_csg-0018_050_max"
   ]
  },
  {
   "job": "CS18_k1_eec0092d",
   "subset": "CS18",
   "k": 1,
   "pages": [
    "e-codices_csg-0018_097_max"
   ]
  },
  {
   "job": "CS18_k3_a4e24e97",
   "subset": "CS18",
   "k": 3,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_108_max",
    "e-codices_csg-0018_156_max"
   ]
  },
  {
   "job": "CS18_k3_f09412e0",
   "subset": "CS18",
   "k": 3,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_118_max",
    "e-codices_csg-0018_150_max"
   ]
  },
  {
   "job": "CS18_k3_4666bf9f",
   "subset": "CS18",
   "k": 3,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_051_max",
    "e-codices_csg-0018_057_max"
   ]
  },
  {
   "job": "CS18_k3_48a31c0d",
   "subset": "CS18",
   "k": 3,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_118_max",
    "e-codices_csg-0018_149_max"
   ]
  },
  {
   "job": "CS18_k3_e5d541e7",
   "subset": "CS18",
   "k": 3,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_097_max",
    "e-codices_csg-0018_115_max"
   ]
  },
  {
   "job": "CS18_k5_815dbac0",
   "subset": "CS18",
   "k": 5,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_108_max",
    "e-codices_csg-0018_153_max",
    "e-codices_csg-0018_155_max",
    "e-codices_csg-0018_156_max"
   ]
  },
  {
   "job": "CS18_k5_4889c850",
   "subset": "CS18",
   "k": 5,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_097_max",
    "e-codices_csg-0018_118_max",
    "e-codices_csg-0018_149_max",
    "e-codices_csg-0018_154_max"
   ]
  },
  {
   "job": "CS18_k5_45a250d0",
   "subset": "CS18",
   "k": 5,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_051_max",
    "e-codices_csg-0018_057_max",
    "e-codices_csg-0018_119_max",
    "e-codices_csg-0018_149_max"
   ]
  },
  {
   "job": "CS18_k5_f9cecfe9",
   "subset": "CS18",
   "k": 5,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_051_max",
    "e-codices_csg-0018_057_max",
    "e-codices_csg-0018_118_max",
    "e-codices_csg-0018_149_max"
   ]
  },
  {
   "job": "CS18_k5_4aa3851f",
   "subset": "CS18",
   "k": 5,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_097_max",
    "e-codices_csg-0018_108_max",
    "e-codices_csg-0018_115_max",
    "e-codices_csg-0018_153_max"
   ]
  },
  {
   "job": "CS18_k10_1e188476",
   "subset": "CS18",
   "k": 10,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_057_max",
    "e-codices_csg-0018_108_max",
    "e-codices_csg-0018_116_max",
    "e-codices_csg-0018_149_max",
    "e-codices_csg-0018_150_max",
    "e-codices_csg-0018_153_max",
    "e-codices_csg-0018_154_max",
    "e-codices_csg-0018_155_max",
    "e-codices_csg-0018_156_max"
   ]
  },
  {
   "job": "CS18_k10_5d8b6b68",
   "subset": "CS18",
   "k": 10,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_051_max",
    "e-codices_csg-0018_097_max",
    "e-codices_csg-0018_099_max",
    "e-codices_csg-0018_116_max",
    "e-codices_csg-0018_118_max",
    "e-codices_csg-0018_119_max",
    "e-codices_csg-0018_149_max",
    "e-codices_csg-0018_153_max",
    "e-codices_csg-0018_154_max"
   ]
  },
  {
   "job": "CS18_k10_d1a49d99",
   "subset": "CS18",
   "k": 10,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_051_max",
    "e-codices_csg-0018_057_max",
    "e-codices_csg-0018_105_max",
    "e-codices_csg-0018_117_max",
    "e-codices_csg-0018_118_max",
    "e-codices_csg-0018_119_max",
    "e-codices_csg-0018_149_max",
    "e-codices_csg-0018_150_max",
    "e-codices_csg-0018_152_max"
   ]
  },
  {
   "job": "CS18_k10_9dc97bff",
   "subset": "CS18",
   "k": 10,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_051_max",
    "e-codices_csg-0018_099_max",
    "e-codices_csg-0018_105_max",
    "e-codices_csg-0018_108_max",
    "e-codices_csg-0018_118_max",
    "e-codices_csg-0018_119_max",
    "e-codices_csg-0018_149_max",
    "e-codices_csg-0018_153_max",
    "e-codices_csg-0018_154_max"
   ]
  },
  {
   "job": "CS18_k10_3049391c",
   "subset": "CS18",
   "k": 10,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_051_max",
    "e-codices_csg-0018_057_max",
    "e-codices_csg-0018_117_max",
    "e-codices_csg-0018_118_max",
    "e-codices_csg-0018_119_max",
    "e-codices_csg-0018_149_max",
    "e-codices_csg-0018_150_max",
    "e-codices_csg-0018_152_max",
    "e-codices_csg-0018_156_max"
   ]
  },
  {
   "job": "CS18_k10_a753e501",
   "subset": "CS18",
   "k": 10,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_051_max",
    "e-codices_csg-0018_057_max",
    "e-codices_csg-0018_097_max",
    "e-codices_csg-0018_108_max",
    "e-codices_csg-0018_115_max",
    "e-codices_csg-0018_117_max",
    "e-codices_csg-0018_118_max",
    "e-codices_csg-0018_153_max",
    "e-codices_csg-0018_154_max"
   ]
  },
  {
   "job": "CS18_k15_8a9edb1c",
   "subset": "CS18",
   "k": 15,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_057_max",
    "e-codices_csg-0018_099_max",
    "e-codices_csg-0018_104_max",
    "e-codices_csg-0018_105_max",
    "e-codices_csg-0018_108_max",
    "e-codices_csg-0018_116_max",
    "e-codices_csg-0018_118_max",
    "e-codices_csg-0018_149_max",
    "e-codices_csg-0018_150_max",
    "e-codices_csg-0018_152_max",
    "e-codices_csg-0018_153_max",
    "e-codices_csg-0018_154_max",
    "e-codices_csg-0018_155_max",
    "e-codices_csg-0018_156_max"
   ]
  },
  {
   "job": "CS18_k15_9d9d1f6a",
   "subset": "CS18",
   "k": 15,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_051_max",
    "e-codices_csg-0018_057_max",
    "e-codices_csg-0018_097_max",
    "e-codices_csg-0018_099_max",
    "e-codices_csg-0018_104_max",
    "e-codices_csg-0018_115_max",
    "e-codices_csg-0018_117_max",
    "e-codices_csg-0018_119_max",
    "e-codices_csg-0018_149_max",
    "e-codices_csg-0018_150_max",
    "e-codices_csg-0018_153_max",
    "e-codices_csg-0018_154_max",
    "e-codices_csg-0018_155_max",
    "e-codices_csg-0018_156_max"
   ]
  },
  {
   "job": "CS18_k15_7e55208c",
   "subset": "CS18",
   "k": 15,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_051_max",
    "e-codices_csg-0018_057_max",
    "e-codices_csg-0018_097_max",
    "e-codices_csg-0018_104_max",
    "e-codices_csg-0018_105_max",
    "e-codices_csg-0018_108_max",
    "e-codices_csg-0018_117_max",
    "e-codices_csg-0018_118_max",
    "e-codices_csg-0018_119_max",
    "e-codices_csg-0018_149_max",
    "e-codices_csg-0018_150_max",
    "e-codices_csg-0018_152_max",
    "e-codices_csg-0018_153_max",
    "e-codices_csg-0018_156_max"
   ]
  },
  {
   "job": "CS18_k15_60eb6abb",
   "subset": "CS18",
   "k": 15,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_051_max",
    "e-codices_csg-0018_057_max",
    "e-codices_csg-0018_097_max",
    "e-codices_csg-0018_099_max",
    "e-codices_csg-0018_104_max",
    "e-codices_csg-0018_105_max",
    "e-codices_csg-0018_117_max",
    "e-codices_csg-0018_118_max",
    "e-codices_csg-0018_119_max",
    "e-codices_csg-0018_149_max",
    "e-codices_csg-0018_150_max",
    "e-codices_csg-0018_152_max",
    "e-codices_csg-0018_153_max",
    "e-codices_csg-0018_156_max"
   ]
  },
  {
   "job": "CS18_k15_c0630f5c",
   "subset": "CS18",
   "k": 15,
   "pages": [
    "e-codices_csg-0018_050_max",
    "e-codices_csg-0018_051_max",
    "e-codices_csg-0018_057_max",
    "e-codices_csg-0018_097_max",
    "e-codices_csg-0018_105_max",
    "e-codices_csg-0018_108_max",
    "e-codices_csg-0018_115_max",
    "e-codices_csg-0018_117_max",
    "e-codices_csg-0018_118_max",
    "e-codices_csg-0018_119_max",
    "e-codices_csg-0018_149_max",
    "e-codices_csg-0018_152_max",
    "e-codices_csg-0018_153_max",
    "e-codices_csg-0018_154_max",
    "e-codices_csg-0018_155_max"
   ]
  },
  {
   "job": "CS863_k1_c19c7a4c",
   "subset": "CS863",
   "k": 1,
   "pages": [
    "e-codices_csg-0863_184_max"
   ]
  },
  {
   "job": "CS863_k1_0fd5a0da",
   "subset": "CS863",
   "k": 1,
   "pages": [
    "e-codices_csg-0863_009_max"
   ]
  },
  {
   "job": "CS863_k1_23a82488",
   "subset": "CS863",
   "k": 1,
   "pages": [
    "e-codices_csg-0863_010_max"
   ]
  },
  {
   "job": "CS863_k1_c2389398",
   "subset": "CS863",
   "k": 1,
   "pages": [
    "e-codices_csg-0863_186_max"
   ]
  },
  {
   "job": "CS863_k1_f4f0a4c0",
   "subset": "CS863",
   "k": 1,
   "pages": [
    "e-codices_csg-0863_012_max"
   ]
  },
  {
   "job": "CS863_k3_37b23fac",
   "subset": "CS863",
   "k": 3,
   "pages": [
    "e-codices_csg-0863_184_max",
    "e-codices_csg-0863_186_max",
    "e-codices_csg-0863_212_max"
   ]
  },
  {
   "job": "CS863_k3_a4e975f7",
   "subset": "CS863",
   "k": 3,
   "pages": [
    "e-codices_csg-0863_010_max",
    "e-codices_csg-0863_153_max",
    "e-codices_csg-0863_186_max"
   ]
  },
  {
   "job": "CS863_k3_063a713a",
   "subset": "CS863",
   "k": 3,
   "pages": [
    "e-codices_csg-0863_009_max",
    "e-codices_csg-0863_010_max",
    "e-codices_csg-0863_012_max"
   ]
  },
  {
   "job": "CS863_k3_f6fbf2ad",
   "subset": "CS863",
   "k": 3,
   "pages": [
    "e-codices_csg-0863_011_max",
    "e-codices_csg-0863_155_max",
    "e-codices_csg-0863_186_max"
   ]
  },
  {
   "job": "CS863_k3_59db31e9",
   "subset": "CS863",
   "k": 3,
   "pages": [
    "e-codices_csg-0863_010_max",
    "e-codices_csg-0863_012_max",
    "e-codices_csg-0863_186_max"
   ]
  },
  {
   "job": "CS863_k3_002db31f",
   "subset": "CS863",
   "k": 3,
   "pages": [
    "e-codices_csg-0863_009_max",
    "e-codices_csg-0863_012_max",
    "e-codices_csg-0863_186_max"
   ]
  },
  {
   "job": "CS863_k5_5feb910b",
   "subset": "CS863",
   "k": 5,
   "pages": [
    "e-codices_csg-0863_184_max",
    "e-codices_csg-0863_186_max",
    "e-codices_csg-0863_212_max",
    "e-codices_csg-0863_250_max",
    "e-codices_csg-0863_260_max"
   ]
  },
  {
   "job": "CS863_k5_7cb780c5",
   "subset": "CS863",
   "k": 5,
   "pages": [
    "e-codices_csg-0863_009_max",
    "e-codices_csg-0863_153_max",
    "e-codices_csg-0863_155_max",
    "e-codices_csg-0863_186_max",
    "e-codices_csg-0863_262_max"
   ]
  },
  {
   "job": "CS863_k5_6565430e",
   "subset": "CS863",
   "k": 5,
   "pages": [
    "e-codices_csg-0863_009_max",
    "e-codices_csg-0863_010_max",
    "e-codices_csg-0863_011_max",
    "e-codices_csg-0863_012_max",
    "e-codices_csg-0863_155_max"
   ]
  },
  {
   "job": "CS863_k5_f2677637",
   "subset": "CS863",
   "k": 5,
   "pages": [
    "e-codices_csg-0863_009_max",
    "e-codices_csg-0863_153_max",
    "e-codices_csg-0863_155_max",
    "e-codices_csg-0863_184_max",
    "e-codices_csg-0863_186_max"
   ]
  },
  {
   "job": "CS863_k5_d03a5cb8",
   "subset": "CS863",
   "k": 5,
   "pages": [
    "e-codices_csg-0863_009_max",
    "e-codices_csg-0863_010_max",
    "e-codices_csg-0863_012_max",
    "e-codices_csg-0863_186_max",
    "e-codices_csg-0863_212_max"
   ]
  },
  {
   "job": "CS863_k5_a902634d",
   "subset": "CS863",
   "k": 5,
   "pages": [
    "e-codices_csg-0863_009_max",
    "e-codices_csg-0863_012_max",
    "e-codices_csg-0863_184_max",
    "e-codices_csg-0863_186_max",
    "e-codices_csg-0863_261_max"
   ]
  },
  {
   "job": "CS863_k10_73d6bbc6",
   "subset": "CS863",
   "k": 10,
   "pages": [
    "e-codices_csg-0863_184_max",
    "e-codices_csg-0863_186_max",
    "e-codices_csg-0863_212_max",
    "e-codices_csg-0863_241_max",
    "e-codices_csg-0863_250_max",
    "e-codices_csg-0863_252_max",
    "e-codices_csg-0863_260_max",
    "e-codices_csg-0863_261_max",
    "e-codices_csg-0863_263_max",
    "e-codices_csg-0863_265_max"
   ]
  },
  {
   "job": "CS863_k10_efc33dfa",
   "subset": "CS863",
   "k": 10,
   "pages": [
    "e-codices_csg-0863_010_max",
    "e-codices_csg-0863_011_max",
    "e-codices_csg-0863_153_max",
    "e-codices_csg-0863_154_max",
    "e-codices_csg-0863_155_max",
    "e-codices_csg-0863_184_max",
    "e-codices_csg-0863_186_max",
    "e-codices_csg-0863_225_max",
    "e-codices_csg-0863_252_max",
    "e-codices_csg-0863_262_max"
   ]
  },
  {
   "job": "CS863_k10_ba6c5cfd",
   "subset": "CS863",
   "k": 10,
   "pages": [
    "e-codices_csg-0863_009_max",
    "e-codices_csg-0863_010_max",
    "e-codices_csg-0863_011_max",
    "e-codices_csg-0863_012_max",
    "e-codices_csg-0863_153_max",
    "e-codices_csg-0863_155_max",
    "e-codices_csg-0863_186_max",
    "e-codices_csg-0863_212_max",
    "e-codices_csg-0863_252_max",
    "e-codices_csg-0863_265_max"
   ]
  },
  {
   "job": "CS863_k10_6949b64f",
   "subset": "CS863",
   "k": 10,
   "pages": [
    "e-codices_csg-0863_011_max",
    "e-codices_csg-0863_012_max",
    "e-codices_csg-0863_153_max",
    "e-codices_csg-0863_154_max",
    "e-codices_csg-0863_155_max",
    "e-codices_csg-0863_184_max",
    "e-codices_csg-0863_186_max",
    "e-codices_csg-0863_225_max",
    "e-codices_csg-0863_252_max",
    "e-codices_csg-0863_262_max"
   ]
  },
  {
   "job": "CS863_k10_f95fa471",
   "subset": "CS863",
   "k": 10,
   "pages": [
    "e-codices_csg-0863_009_max",
    "e-codices_csg-0863_010_max",
    "e-codices_csg-0863_011_max",
    "e-codices_csg-0863_012_max",
    "e-codices_csg-0863_153_max",
    "e-codices_csg-0863_155_max",
    "e-codices_csg-0863_186_max",
    "e-codices_csg-0863_212_max",
    "e-codices_csg-0863_225_max",
    "e-codices_csg-0863_252_max"
   ]
  },
  {
   "job": "CS863_k10_b045f2ec",
   "subset": "CS863",
   "k": 10,
   "pages": [
    "e-codices_csg-0863_009_max",
    "e-codices_csg-0863_010_max",
    "e-codices_csg-0863_011_max",
    "e-codices_csg-0863_012_max",
    "e-codices_csg-0863_184_max",
    "e-codices_csg-0863_186_max",
    "e-codices_csg-0863_225_max",
    "e-codices_csg-0863_241_max",
    "e-codices_csg-0863_261_max",
    "e-codices_csg-0863_262_max"
   ]
  },
  {
   "job": "CS863_k15_02e6fe88",
   "subset": "CS863",
   "k": 15,
   "pages": [
    "e-codices_csg-0863_012_max",
    "e-codices_csg-0863_154_max",
    "e-codices_csg-0863_184_max",
    "e-codices_csg-0863_186_max",
    "e-codices_csg-0863_212_max",
    "e-codices_csg-0863_225_max",
    "e-codices_csg-0863_241_max",
    "e-codices_csg-0863_250_max",
    "e-codices_csg-0863_252_max",
    "e-codices_csg-0863_257_max",
    "e-codices_csg-0863_260_max",
    "e-codices_csg-0863_261_max",
    "e-codices_csg-0863_262_max",
    "e-codices_csg-0863_263_max",
    "e-codices_csg-0863_265_max"
   ]
  },
  {
   "job": "CS863_k15_eb326e37",
   "subset": "CS863",
   "k": 15,
   "pages": [
    "e-codices_csg-0863_009_max",
    "e-codices_csg-0863_010_max",
    "e-codices_csg-0863_011_max",
    "e-codices_csg-0863_153_max",
    "e-codices_csg-0863_154_max",
    "e-codices_csg-0863_155_max",
    "e-codices_csg-0863_184_max",
    "e-codices_csg-0863_186_max",
    "e-codices_csg-0863_212_max",
    "e-codices_csg-0863_225_max",
    "e-codices_csg-0863_241_max",
    "e-codices_csg-0863_252_max",
    "e-codices_csg-0863_260_max",
    "e-codices_csg-0863_262_max",
    "e-codices_csg-0863_265_max"
   ]
  },
  {
   "job": "CS863_k15_8895bb47",
   "subset": "CS863",
   "k": 15,
   "pages": [
    "e-codices_csg-0863_009_max",
    "e-codices_csg-0863_010_max",
    "e-codices_csg-0863_011_max",
    "e-codices_csg-0863_012_max",
    "e-codices_csg-0863_153_max",
    "e-codices_csg-0863_154_max",
    "e-codices_csg-0863_155_max",
    "e-codices_csg-0863_184_max",
    "e-codices_csg-0863_186_max",
    "e-codices_csg-0863_212_max",
    "e-codices_csg-0863_225_max",
    "e-codices_csg-0863_250_max",
    "e-codices_csg-0863_252_max",
    "e-codices_csg-0863_257_max",
    "e-codices_csg-0863_265_max"
   ]
  },
  {
   "job": "CS863_k15_f84a921c",
   "subset": "CS863",
   "k": 15,
   "pages": [
    "e-codices_csg-0863_009_max",
    "e-codices_csg-0863_011_max",
    "e-codices_csg-0863_012_max",
    "e-codices_csg-0863_153_max",
    "e-codices_csg-0863_154_max",
    "e-codices_csg-0863_155_max",
    "e-codices_csg-0863_184_max",
    "e-codices_csg-0863_186_max",
    "e-codices_csg-0863_212_max",
    "e-codices_csg-0863_225_max",
    "e-codices_csg-0863_252_max",
    "e-codices_csg-0863_260_max",
    "e-codices_csg-0863_261_max",
    "e-codices_csg-0863_262_max",
    "e-codices_csg-0863_265_max"
   ]
  },
  {
   "job": "CS863_k15_351ed81e",
   "subset": "CS863",
   "k": 15,
   "pages": [
    "e-codices_csg-0863_009_max",
    "e-codices_csg-0863_010_max",
    "e-codices_csg-0863_011_max",
    "e-codices_csg-0863_012_max",
    "e-codices_csg-0863_153_max",
    "e-codices_csg-0863_154_max",
    "e-codices_csg-0863_155_max",
    "e-codices_csg-0863_184_max",
    "e-codices_csg-0863_186_max",
    "e-codices_csg-0863_212_max",
    "e-codices_csg-0863_225_max",
    "e-codices_csg-0863_252_max",
    "e-codices_csg-0863_257_max",
    "e-codices_csg-0863_263_max",
    "e-codices_csg-0863_265_max"
   ]
  },
  {
   "job": "CS863_k15_a02442ed",
   "subset": "CS863",
   "k": 15,
   "pages": [
    "e-codices_csg-0863_009_max",
    "e-codices_csg-0863_010_max",
    "e-codices_csg-0863_011_max",
    "e-codices_csg-0863_012_max",
    "e-codices_csg-0863_155_max",
    "e-codices_csg-0863_184_max",
    "e-codices_csg-0863_186_max",
    "e-codices_csg-0863_225_max",
    "e-codices_csg-0863_241_max",
    "e-codices_csg-0863_250_max",
    "e-codices_csg-0863_252_max",
    "e-codices_csg-0863_260_max",
    "e-codices_csg-0863_261_max",
    "e-codices_csg-0863_262_max",
    "e-codices_csg-0863_263_max"
   ]
  }
 ],
 "cells": [
  {
   "subset": "CB55",
   "method": "grayscale_variance",
   "k": 1,
   "job": "CB55_k1_8c26820d"
  },
  {
   "subset": "CB55",
   "method": "pca_max_distance",
   "k": 1,
   "job": "CB55_k1_7a1608db"
  },
  {
   "subset": "CB55",
   "method": "pca_centroid",
   "k": 1,
   "job": "CB55_k1_ddbc80ee"
  },
  {
   "subset": "CB55",
   "method": "ica_max_distance",
   "k": 1,
   "job": "CB55_k1_7a1608db"
  },
  {
   "subset": "CB55",
   "method": "ica_centroid",
   "k": 1,
   "job": "CB55_k1_ddbc80ee"
  },
  {
   "subset": "CB55",
   "method": "random",
   "k": 1,
   "job": "CB55_k1_2c752f4a"
  },
  {
   "subset": "CB55",
   "method": "grayscale_variance",
   "k": 3,
   "job": "CB55_k3_e5547c3a"
  },
  {
   "subset": "CB55",
   "method": "pca_max_distance",
   "k": 3,
   "job": "CB55_k3_f2700177"
  },
  {
   "subset": "CB55",
   "method": "pca_centroid",
   "k": 3,
   "job": "CB55_k3_ce938512"
  },
  {
   "subset": "CB55",
   "method": "ica_max_distance",
   "k": 3,
   "job": "CB55_k3_2017fce1"
  },
  {
   "subset": "CB55",
   "method": "ica_centroid",
   "k": 3,
   "job": "CB55_k3_458ebb5d"
  },
  {
   "subset": "CB55",
   "method": "random",
   "k": 3,
   "job": "CB55_k3_9c4b2a61"
  },
  {
   "subset": "CB55",
   "method": "grayscale_variance",
   "k": 5,
   "job": "CB55_k5_4d2a742b"
  },
  {
   "subset": "CB55",
   "method": "pca_max_distance",
   "k": 5,
   "job": "CB55_k5_86a6d5af"
  },
  {
   "subset": "CB55",
   "method": "pca_centroid",
   "k": 5,
   "job": "CB55_k5_82d0bdca"
  },
  {
   "subset": "CB55",
   "method": "ica_max_distance",
   "k": 5,
   "job": "CB55_k5_9072af77"
  },
  {
   "subset": "CB55",
   "method": "ica_centroid",
   "k": 5,
   "job": "CB55_k5_45043204"
  },
  {
   "subset": "CB55",
   "method": "random",
   "k": 5,
   "job": "CB55_k5_87c37879"
  },
  {
   "subset": "CB55",
   "method": "grayscale_variance",
   "k": 10,
   "job": "CB55_k10_69b45a69"
  },
  {
   "subset": "CB55",
   "method": "pca_max_distance",
   "k": 10,
   "job": "CB55_k10_d688d043"
  },
  {
   "subset": "CB55",
   "method": "pca_centroid",
   "k": 10,
   "job": "CB55_k10_5a5195ef"
  },
  {
   "subset": "CB55",
   "method": "ica_max_distance",
   "k": 10,
   "job": "CB55_k10_ad7bd714"
  },
  {
   "subset": "CB55",
   "method": "ica_centroid",
   "k": 10,
   "job": "CB55_k10_5a5195ef"
  },
  {
   "subset": "CB55",
   "method": "random",
   "k": 10,
   "job": "CB55_k10_69341241"
  },
  {
   "subset": "CB55",
   "method": "grayscale_variance",
   "k": 15,
   "job": "CB55_k15_668f4fda"
  },
  {
   "subset": "CB55",
   "method": "pca_max_distance",
   "k": 15,
   "job": "CB55_k15_5c363272"
  },
  {
   "subset": "CB55",
   "method": "pca_centroid",
   "k": 15,
   "job": "CB55_k15_634568b4"
  },
  {
   "subset": "CB55",
   "method": "ica_max_distance",
   "k": 15,
   "job": "CB55_k15_5c363272"
  },
  {
   "subset": "CB55",
   "method": "ica_centroid",
   "k": 15,
   "job": "CB55_k15_634568b4"
  },
  {
   "subset": "CB55",
   "method": "random",
   "k": 15,
   "job": "CB55_k15_7e3c513c"
  },
  {
   "subset": "CS18",
   "method": "grayscale_variance",
   "k": 1,
   "job": "CS18_k1_789b4135"
  },
  {
   "subset": "CS18",
   "method": "pca_max_distance",
   "k": 1,
   "job": "CS18_k1_789b4135"
  },
  {
   "subset": "CS18",
   "method": "pca_centroid",
   "k": 1,
   "job": "CS18_k1_789b4135"
  },
  {
   "subset": "CS18",
   "method": "ica_max_distance",
   "k": 1,
   "job": "CS18_k1_789b4135"
  },
  {
   "subset": "CS18",
   "method": "ica_centroid",
   "k": 1,
   "job": "CS18_k1_789b4135"
  },
  {
   "subset": "CS18",
   "method": "random",
   "k": 1,
   "job": "CS18_k1_eec0092d"
  },
  {
   "subset": "CS18",
   "method": "grayscale_variance",
   "k": 3,
   "job": "CS18_k3_a4e24e97"
  },
  {
   "subset": "CS18",
   "method": "pca_max_distance",
   "k": 3,
   "job": "CS18_k3_f09412e0"
  },
  {
   "subset": "CS18",
   "method": "pca_centroid",
   "k": 3,
   "job": "CS18_k3_4666bf9f"
  },
  {
   "subset": "CS18",
   "method": "ica_max_distance",
   "k": 3,
   "job": "CS18_k3_48a31c0d"
  },
  {
   "subset": "CS18",
   "method": "ica_centroid",
   "k": 3,
   "job": "CS18_k3_4666bf9f"
  },
  {
   "subset": "CS18",
   "method": "random",
   "k": 3,
   "job": "CS18_k3_e5d541e7"
  },
  {
   "subset": "CS18",
   "method": "grayscale_variance",
   "k": 5,
   "job": "CS18_k5_815dbac0"
  },
  {
   "subset": "CS18",
   "method": "pca_max_distance",
   "k": 5,
   "job": "CS18_k5_4889c850"
  },
  {
   "subset": "CS18",
   "method": "pca_centroid",
   "k": 5,
   "job": "CS18_k5_45a250d0"
  },
  {
   "subset": "CS18",
   "method": "ica_max_distance",
   "k": 5,
   "job": "CS18_k5_4889c850"
  },
  {
   "subset": "CS18",
   "method": "ica_centroid",
   "k": 5,
   "job": "CS18_k5_f9cecfe9"
  },
  {
   "subset": "CS18",
   "method": "random",
   "k": 5,
   "job": "CS18_k5_4aa3851f"
  },
  {
   "subset": "CS18",
   "method": "grayscale_variance",
   "k": 10,
   "job": "CS18_k10_1e188476"
  },
  {
   "subset": "CS18",
   "method": "pca_max_distance",
   "k": 10,
   "job": "CS18_k10_5d8b6b68"
  },
  {
   "subset": "CS18",
   "method": "pca_centroid",
   "k": 10,
   "job": "CS18_k10_d1a49d99"
  },
  {
   "subset": "CS18",
   "method": "ica_max_distance",
   "k": 10,
   "job": "CS18_k10_9dc97bff"
  },
  {
   "subset": "CS18",
   "method": "ica_centroid",
   "k": 10,
   "job": "CS18_k10_3049391c"
  },
  {
   "subset": "CS18",
   "method": "random",
   "k": 10,
   "job": "CS18_k10_a753e501"
  },
  {
   "subset": "CS18",
   "method": "grayscale_variance",
   "k": 15,
   "job": "CS18_k15_8a9edb1c"
  },
  {
   "subset": "CS18",
   "method": "pca_max_distance",
   "k": 15,
   "job": "CS18_k15_9d9d1f6a"
  },
  {
   "subset": "CS18",
   "method": "pca_centroid",
   "k": 15,
   "job": "CS18_k15_7e55208c"
  },
  {
   "subset": "CS18",
   "method": "ica_max_distance",
   "k": 15,
   "job": "CS18_k15_9d9d1f6a"
  },
  {
   "subset": "CS18",
   "method": "ica_centroid",
   "k": 15,
   "job": "CS18_k15_60eb6abb"
  },
  {
   "subset": "CS18",
   "method": "random",
   "k": 15,
   "job": "CS18_k15_c0630f5c"
  },
  {
   "subset": "CS863",
   "method": "grayscale_variance",
   "k": 1,
   "job": "CS863_k1_c19c7a4c"
  },
  {
   "subset": "CS863",
   "method": "pca_max_distance",
   "k": 1,
   "job": "CS863_k1_0fd5a0da"
  },
  {
   "subset": "CS863",
   "method": "pca_centroid",
   "k": 1,
   "job": "CS863_k1_23a82488"
  },
  {
   "subset": "CS863",
   "method": "ica_max_distance",
   "k": 1,
   "job": "CS863_k1_0fd5a0da"
  },
  {
   "subset": "CS863",
   "method": "ica_centroid",
   "k": 1,
   "job": "CS863_k1_c2389398"
  },
  {
   "subset": "CS863",
   "method": "random",
   "k": 1,
   "job": "CS863_k1_f4f0a4c0"
  },
  {
   "subset": "CS863",
   "method": "grayscale_variance",
   "k": 3,
   "job": "CS863_k3_37b23fac"
  },
  {
   "subset": "CS863",
   "method": "pca_max_distance",
   "k": 3,
   "job": "CS863_k3_a4e975f7"
  },
  {
   "subset": "CS863",
   "method": "pca_centroid",
   "k": 3,
   "job": "CS863_k3_063a713a"
  },
  {
   "subset": "CS863",
   "method": "ica_max_distance",
   "k": 3,
   "job": "CS863_k3_f6fbf2ad"
  },
  {
   "subset": "CS863",
   "method": "ica_centroid",
   "k": 3,
   "job": "CS863_k3_59db31e9"
  },
  {
   "subset": "CS863",
   "method": "random",
   "k": 3,
   "job": "CS863_k3_002db31f"
  },
  {
   "subset": "CS863",
   "method": "grayscale_variance",
   "k": 5,
   "job": "CS863_k5_5feb910b"
  },
  {
   "subset": "CS863",
   "method": "pca_max_distance",
   "k": 5,
   "job": "CS863_k5_7cb780c5"
  },
  {
   "subset": "CS863",
   "method": "pca_centroid",
   "k": 5,
   "job": "CS863_k5_6565430e"
  },
  {
   "subset": "CS863",
   "method": "ica_max_distance",
   "k": 5,
   "job": "CS863_k5_f2677637"
  },
  {
   "subset": "CS863",
   "method": "ica_centroid",
   "k": 5,
   "job": "CS863_k5_d03a5cb8"
  },
  {
   "subset": "CS863",
   "method": "random",
   "k": 5,
   "job": "CS863_k5_a902634d"
  },
  {
   "subset": "CS863",
   "method": "grayscale_variance",
   "k": 10,
   "job": "CS863_k10_73d6bbc6"
  },
  {
   "subset": "CS863",
   "method": "pca_max_distance",
   "k": 10,
   "job": "CS863_k10_efc33dfa"
  },
  {
   "subset": "CS863",
   "method": "pca_centroid",
   "k": 10,
   "job": "CS863_k10_ba6c5cfd"
  },
  {
   "subset": "CS863",
   "method": "ica_max_distance",
   "k": 10,
   "job": "CS863_k10_6949b64f"
  },
  {
   "subset": "CS863",
   "method": "ica_centroid",
   "k": 10,
   "job": "CS863_k10_f95fa471"
  },
  {
   "subset": "CS863",
   "method": "random",
   "k": 10,
   "job": "CS863_k10_b045f2ec"
  },
  {
   "subset": "CS863",
   "method": "grayscale_variance",
   "k": 15,
   "job": "CS863_k15_02e6fe88"
  },
  {
   "subset": "CS863",
   "method": "pca_max_distance",
   "k": 15,
   "job": "CS863_k15_eb326e37"
  },
  {
   "subset": "CS863",
   "method": "pca_centroid",
   "k": 15,
   "job": "CS863_k15_8895bb47"
  },
  {
   "subset": "CS863",
   "method": "ica_max_distance",
   "k": 15,
   "job": "CS863_k15_f84a921c"
  },
  {
   "subset": "CS863",
   "method": "ica_centroid",
   "k": 15,
   "job": "CS863_k15_351ed81e"
  },
  {
   "subset": "CS863",
   "method": "random",
   "k": 15,
   "job": "CS863_k15_a02442ed"
  }
 ]
}""")
JOBS = [dict(tag=t, encoder=e, init=i, subset=m["subset"], k=m["k"], pages=m["pages"], job=m["job"],
             name=f"rq2_unet_{i}_{m['job']}")
        for (t, e, i) in CONFIGS for m in MANIFEST["jobs"]]
print(len(MANIFEST["jobs"]), "page sets x", len(CONFIGS), "initializations =", len(JOBS), "runs")

def sh(cmd, check=True, **kw):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check, **kw)

## 2. Drive and GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
(DRIVE_OUT / "runs").mkdir(exist_ok=True)
(DRIVE_OUT / "results").mkdir(exist_ok=True)
(DRIVE_OUT / "logs").mkdir(exist_ok=True)

import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
import psutil
ram, cpus = psutil.virtual_memory().total / 2**30, os.cpu_count()
if PARALLEL is None:
    PARALLEL = max(1, min(int(0.9 * vram // GB_PER_RUN), int(0.85 * ram // RAM_PER_RUN), len(JOBS)))
THREADS = max(1, cpus // PARALLEL)   # CPU threads per run (pages are cached, no loader workers)
print(f"{torch.cuda.get_device_name(0)}: {vram:.0f} GiB VRAM, {ram:.0f} GiB RAM, {cpus} CPU cores"
      f" -> {PARALLEL} parallel runs")

## 3. Dependencies (Python packages and Java for the DIVA evaluator)

In [ ]:
sh('pip -q install "timm==1.0.24" "segmentation-models-pytorch==0.5.0" einops pyyaml '
   'opencv-python-headless scikit-image scikit-learn shapely')
sh("apt-get -qq update > /dev/null && apt-get -qq install -y default-jre-headless openjfx > /dev/null")
sh("java -version")
if not JAR.exists():
    sh(f"curl -sL -o {JAR} {JAR_URL}")
assert JAR.stat().st_size > 1_000_000, "DIVA evaluator jar download failed"
JAVA_CP = f"/usr/share/openjfx/lib/*:{JAR}"
print("evaluator:", JAR, JAR.stat().st_size, "bytes")

## 4. Project code (embedded snapshot of the thesis repository)

In [ ]:
BUNDLE_B64 = (
    "..."
)

import base64, io, tarfile
REPO.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE_B64)), mode="r:gz") as tar:
    tar.extractall(REPO)
STAGE = REPO / "50_modelling" / "01_simple_segmentation"

import yaml
base = yaml.safe_load(open(STAGE / "configs/unet_resnet_diva.yaml"))
c = json.loads(json.dumps(base))
c["model"].update(encoder_weights="imagenet", pretrained=True, freeze_backbone=False)
c["training"]["vis_every"] = 0   # visualisations would re-read full validation pages
yaml.safe_dump(c, open(STAGE / "configs/unet_diva_rq2.yaml", "w"), sort_keys=False)
print("code ready:", sorted(p.name for p in STAGE.iterdir()))

## 5. DIVA-HisDB from Zenodo

Downloads the 1.3 GB archive once, extracts only the images, pixel-level ground truth and Task-2 PAGE XML of the three manuscripts, and checks the page counts (20/10/10 per manuscript).

In [ ]:
import tarfile, zipfile
ARCHIVE = Path("/content/DIVAHisDB.tar.gz")
PREFIX = "hisdoc/sites/diuf.unifr.ch.main.hisdoc/files/uploads/diva-hisdb/hisdoc/"

def ready():
    try:
        return all(len(list((DATA / s / f"img-{s}/img" / sp).glob("*.jpg"))) == n
                   for s in SUBSETS for sp, n in (("training", 20), ("validation", 10), ("public-test", 10)))
    except FileNotFoundError:
        return False

if not ready():
    if not ARCHIVE.exists():
        sh(f"curl -sL -o {ARCHIVE} {ZENODO}")
    wanted = {}
    for s in SUBSETS:
        task2 = "CSG863" if s == "CS863" else s
        wanted[f"img-{s}.zip"] = DATA / s / f"img-{s}"
        wanted[f"pixel-level-gt-{s}.zip"] = DATA / s / f"pixel-level-gt-{s}"
        wanted[f"PAGE-gt-{task2}-TASK-2.zip"] = DATA / s / f"PAGE-gt-{s}-TASK-2"
    tmp = Path("/content/diva_zips"); tmp.mkdir(exist_ok=True)
    with tarfile.open(ARCHIVE) as tar:
        for name in wanted:
            member = tar.getmember(PREFIX + name)
            with tar.extractfile(member) as src, open(tmp / name, "wb") as dst:
                shutil.copyfileobj(src, dst)
    for name, target in wanted.items():
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(tmp / name) as z:
            z.extractall(target)
    shutil.rmtree(tmp)

assert ready(), "DIVA-HisDB layout check failed"
for s in SUBSETS:
    n_gt = len(list((DATA / s / f"pixel-level-gt-{s}/pixel-level-gt").rglob("*.png")))
    n_xml = len(list((DATA / s / f"PAGE-gt-{s}-TASK-2/TASK-2").rglob("*.xml")))
    print(s, "images ok, pixel GT:", n_gt, "PAGE XML:", n_xml)

## 6. Encoder check

Builds each configuration once (this also downloads the pretrained weights) and verifies that the pretrained initializations really load their checkpoint, while the random ones do not.

In [ ]:
import copy, torch, timm
sys.path.insert(0, str(STAGE))
os.chdir(STAGE)
from models import build_model

REF = {("tu-convnext_tiny.in12k_ft_in1k", "imagenet"): "convnext_tiny.in12k_ft_in1k",
       ("tu-convnext_tiny.dinov3_lvd1689m", "dinov3"): "convnext_tiny.dinov3_lvd1689m"}

def cfg_for(encoder, init):
    c = yaml.safe_load(open(STAGE / "configs/unet_diva_rq2.yaml"))
    c["model"]["encoder_name"] = encoder
    return c

for tag, enc, init in CONFIGS:
    net = build_model(cfg_for(enc, init)).eval()
    sd = net.state_dict()
    n = sum(p.numel() for p in net.parameters()) / 1e6
    status = "random"
    if init != "random":
        ref = timm.create_model(REF[(enc, init)], pretrained=True, num_classes=0).state_dict()
        stem = [v for k, v in ref.items() if k.endswith("stem.0.weight")]
        ok = any(a.shape == b.shape and torch.allclose(a.float(), b.float()) for b in stem for a in sd.values())
        assert ok, f"{enc}/{init}: pretrained weights NOT loaded"
        status = "pretrained weights verified"
    print(f"{tag:14s} {init:9s} {n:6.1f}M params  {status}")
    del net
torch.cuda.empty_cache()

## 7. Train and evaluate

Each run: train (log to Drive) → copy `best.pth` to Drive → seam-carving postprocessing + DIVA evaluator on the public test split → `results/<run>.json` on Drive. Finished runs are skipped; a checkpoint already on Drive is re-used without retraining.

In [ ]:
import csv, time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed

LOCAL_RUNS = Path("/content/runs")
LOCAL_RUNS.mkdir(exist_ok=True)
lock = threading.Lock()

def log(msg):
    with lock:
        print(time.strftime("%H:%M:%S"), msg, flush=True)

def pids(*parts):
    """PIDs of live processes whose command line contains all parts."""
    out = []
    for d in Path("/proc").iterdir():
        if not d.name.isdigit():
            continue
        try:
            cmd = (d / "cmdline").read_bytes().replace(b"\0", b" ").decode(errors="ignore")
            state = (d / "stat").read_text().rsplit(")", 1)[1].split()[0]
        except OSError:
            continue
        if state != "Z" and "python" in cmd.split(" ", 1)[0] and all(p in cmd for p in parts):
            out.append(int(d.name))
    return out

def job_pids(name):
    return pids("train.py", f"--run-name {name} ") + pids("predict_diva_seamcarve.py", f"/{name}/best.pth")

def wait_for(ps):
    while any(p in pids() for p in ps):
        time.sleep(30)

def run_job(j):
    name = j["name"]
    res = DRIVE_OUT / "results" / f"{name}.json"
    if res.exists():
        return name, "skip (done)"
    logf = DRIVE_OUT / "logs" / f"{name}.log"
    ck_local = LOCAL_RUNS / name / "best.pth"
    ck_drive = DRIVE_OUT / "runs" / name / "best.pth"
    env = {**os.environ, "OMP_NUM_THREADS": str(THREADS), "MKL_NUM_THREADS": str(THREADS)}
    t0 = time.time()
    alive = job_pids(name)
    if alive:   # still running from an interrupted execution -> wait instead of restarting
        log(f"waiting for {name} (pid {alive}) from previous execution")
        wait_for(alive)
    last_attempt = logf.read_text(errors="ignore").split("Seed: ")[-1] if logf.exists() else ""
    if ck_drive.exists():   # training finished earlier -> evaluate only
        if not ck_local.exists():
            ck_local.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(ck_drive, ck_local)
    elif ck_local.exists() and "Training complete" in last_attempt:
        ck_drive.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(ck_local, ck_drive)
    else:                   # (re)train from scratch; drop any interrupted local run
        shutil.rmtree(LOCAL_RUNS / name, ignore_errors=True)
        log(f"train {name}")
        cmd = (f"{sys.executable} train.py --config configs/unet_diva_rq2.yaml --encoder {j['encoder']}"
               f" --manuscript {j['subset']} --k-shot {j['k']} --pages {','.join(j['pages'])}"
               f" --epochs {EPOCHS} --patience {PATIENCE} --lr {LR} --lr-backbone {LR_BACKBONE}"
               f" --batch-size {BATCH_SIZE} --weight-decay {WEIGHT_DECAY} --seed {SEED}"
               f" --val-every {VAL_EVERY} --val-window-batch {VAL_WINDOW_BATCH} --val-bf16 --cache-pages"
               f" --run-name {name} --out-dir {LOCAL_RUNS}")
        with open(logf, "a") as fh:
            r = subprocess.run(cmd, shell=True, cwd=STAGE, stdout=fh, stderr=subprocess.STDOUT, env=env,
                               start_new_session=True)   # survives a cell interrupt
        if r.returncode != 0 or not ck_local.exists():
            return name, f"FAILED training (see {logf})"
        ck_drive.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(ck_local, ck_drive)
    out = LOCAL_RUNS / name / "eval"
    summary = out / "diva_summary.csv"
    if not summary.exists() or summary.stat().st_mtime < ck_local.stat().st_mtime:
        log(f"evaluate {name}")
        cmd = (f"{sys.executable} predict_diva_seamcarve.py --checkpoint {ck_local} --manuscript {j['subset']}"
               f" --split test --output-dir {out} --approx-ratio 0.001 --java-cp '{JAVA_CP}'")
        with open(logf, "a") as fh:
            r = subprocess.run(cmd, shell=True, cwd=STAGE, stdout=fh, stderr=subprocess.STDOUT, env=env,
                               start_new_session=True)   # survives a cell interrupt
        if r.returncode != 0 or not summary.exists():
            return name, f"FAILED evaluation (see {logf})"
    row = next(csv.DictReader(open(summary)))
    keys = {"PixelIU": "Pixel_IU", "LinesIU": "Line_IU", "LinesRecall": "DR", "LinesPrecision": "RA", "LinesFMeasure": "FM"}
    metrics = {v: float(row[k]) for k, v in keys.items()}
    shutil.copytree(out, DRIVE_OUT / "runs" / name / "eval", dirs_exist_ok=True)
    shutil.rmtree(LOCAL_RUNS / name / "eval", ignore_errors=True)   # keep the VM disk small
    res.write_text(json.dumps({**{k: j[k] for k in ("tag", "encoder", "init", "subset", "k", "job", "name")}, **metrics,
                               "minutes": round((time.time() - t0) / 60, 1)}, indent=2))
    return name, "ok FM=%.2f" % (100 * metrics["FM"])

todo = [j for j in JOBS if not (DRIVE_OUT / "results" / f"{j['name']}.json").exists()]
print(f"{len(JOBS) - len(todo)} finished, {len(todo)} to run, {PARALLEL} in parallel")
todo.sort(key=lambda j: not job_pids(j["name"]))   # runs still alive first
pool = ThreadPoolExecutor(max_workers=PARALLEL)
try:
    for fut in as_completed([pool.submit(run_job, j) for j in todo]):
        name, status = fut.result()
        log(f"[{status}] {name}")
    pool.shutdown()
except KeyboardInterrupt:
    # Queued runs are cancelled; runs already started keep going in the background and are
    # picked up (waited for, then evaluated) by the next execution of this cell.
    pool.shutdown(wait=False, cancel_futures=True)
    print("Interrupted: queue cancelled, running trainings continue:",
          [j["name"] for j in todo if job_pids(j["name"])])

## 8. Results

Expands the 77 page-set runs back to the 90 (subset, method, k) cells and writes `MyDrive/thesis_rq2_unet_diva/rq2_unet_cells.csv` (metrics as fractions). The table shows the mean FM (%) over CB55, CS18 and CS863 where all three subsets are finished.

In [ ]:
import pandas as pd
runs = {(r["init"], r["job"]): r for r in (json.loads(p.read_text()) for p in (DRIVE_OUT / "results").glob("*.json"))}
rows = []
for c in MANIFEST["cells"]:
    for (_, _, init) in CONFIGS:
        r = runs.get((init, c["job"]))
        if r:
            rows.append({**c, "init": init, **{m: r[m] for m in ("Pixel_IU", "Line_IU", "DR", "RA", "FM")}})
df = pd.DataFrame(rows)
df.to_csv(DRIVE_OUT / "rq2_unet_cells.csv", index=False)
print(f"{len(runs)}/{len(JOBS)} runs finished, {len(df)}/{len(MANIFEST['cells']) * len(CONFIGS)} cells"
      f" -> {DRIVE_OUT / 'rq2_unet_cells.csv'}")
if len(df):
    g = df.groupby(["method", "init", "k"])
    mean = (100 * g["FM"].mean()).where(g["subset"].nunique() == len(SUBSETS)).round(2)
    display(mean.unstack("k"))